# SmartTrip AI — Kaggle : 3 méthodes + Streamlit
Exécuter les cellules dans l’ordre. Ajouter les deux CSV dans **Add Input** de Kaggle et activer **Internet** pour le tunnel Streamlit. Aucune destination n’est fournie par l’utilisateur.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

## Cellule 1 — Fusionner les deux datasets

In [8]:
#budget = pd.read_csv(find_csv("smarttrip_ai_final_clean.csv"))
#budget.head

In [9]:


def find_csv(name):
    files = list(Path("/kaggle/input").rglob(name))
    if not files:
        raise FileNotFoundError(f"Ajoute {name} dans Kaggle > Add Input")
    return files[0]

dest = pd.read_csv(find_csv("destinations_extended_corrected.csv"))
budget = pd.read_csv(find_csv("smarttrip_ai_final_clean.csv"))

df = dest.merge(
    budget[["country", "destination", "budget_usd_day"]],
    on=["country", "destination"],
    how="left"
)

activities = [
    "culture_score",
    "nature_score",
    "adventure_score",
    "gastronomy_score",
    "nightlife_score"
]

df = df.dropna(subset=activities + ["budget_usd_day"])
print("Dataset fusionné :", df.shape)
display(df.head())

Dataset fusionné : (1020, 18)


,country,destination,culture_score,nature_score,adventure_score,nightlife_score,gastronomy_score,total_activities,latitude,longitude,month,avg_temp_c,rainfall_mm,season,favorable,climate_type,climate_description,budget_usd_day
0,Algeria,Algiers,29.68,10.19,0.25,0.25,56.56,1570,36.746421,3.072833,1,11.83,66.60,Winter,False,Csa,Hot-summer Mediterranean,26.4
1,Algeria,Algiers,29.68,10.19,0.25,0.25,56.56,1570,36.746421,3.072833,2,12.73,51.72,Winter,False,Csa,Hot-summer Mediterranean,26.4
2,Algeria,Algiers,29.68,10.19,0.25,0.25,56.56,1570,36.746421,3.072833,3,14.55,50.84,Spring,False,Csa,Hot-summer Mediterranean,26.4
3,Algeria,Algiers,29.68,10.19,0.25,0.25,56.56,1570,36.746421,3.072833,4,16.43,52.34,Spring,False,Csa,Hot-summer Mediterranean,26.4
4,Algeria,Algiers,29.68,10.19,0.25,0.25,56.56,1570,36.746421,3.072833,5,19.28,57.24,Spring,True,Csa,Hot-summer Mediterranean,26.4


## Cellule 2 — Construire les trois méthodes sur tout le dataset
Cette cellule crée `recommender.py` depuis le notebook. Les trois méthodes utilisent toutes les lignes du dataset, pas une destination de départ.

In [10]:
# CELLULE 2 — MODELING DES 3 MÉTHODES

import numpy as np
import joblib
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.neighbors import NearestNeighbors

# =====================================
# 1. PRÉPARATION DES DONNÉES
# =====================================

features = activities + [
    "avg_temp_c",
    "budget_usd_day"
]

df = df.dropna(subset=features).reset_index(drop=True)

scaler = MinMaxScaler()
X = scaler.fit_transform(df[features])

print("Dataset :", X.shape)

# =====================================
# 2. MÉTHODE 1 : WEIGHTED SCORE
# =====================================

# Préparer les caractéristiques et les poids
# Le score final sera calculé selon les
# préférences de l'utilisateur.

weighted_model = {
    "features": features,
    "vectors": X,
    "weights": {
        "activities": 0.70,
        "climate": 0.20,
        "budget": 0.10
    }
}

print("✅ Weighted Score préparé")

# =====================================
# 3. MÉTHODE 2 : COSINE SIMILARITY
# =====================================

# Construire les vecteurs normalisés
# de TOUTES les destinations

cosine_vectors = normalize(X, norm="l2")

cosine_model = {
    "features": features,
    "vectors": cosine_vectors
}

print("✅ Cosine Similarity préparé")

# =====================================
# 4. MÉTHODE 3 : KNN
# =====================================

# Entraînement / ajustement sur
# TOUTES les destinations

knn = NearestNeighbors(
    metric="manhattan",
    algorithm="brute"
)

knn.fit(X)

knn_model = {
    "features": features,
    "model": knn
}

print("✅ KNN entraîné")

# =====================================
# 5. SAUVEGARDE DES 3 MÉTHODES
# =====================================

model_dir = Path("/kaggle/working/models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    weighted_model,
    model_dir / "weighted_score.joblib"
)

joblib.dump(
    cosine_model,
    model_dir / "cosine_similarity.joblib"
)

joblib.dump(
    knn_model,
    model_dir / "knn.joblib"
)

# Sauvegarder aussi les données et le scaler
joblib.dump(
    {
        "df": df,
        "X": X,
        "scaler": scaler,
        "knn": knn
    },
    model_dir / "smarttrip.joblib"
)

print("\n🎉 LES 3 MÉTHODES SONT PRÊTES ET SAUVEGARDÉES !")

Dataset : (1020, 7)
✅ Weighted Score préparé
✅ Cosine Similarity préparé
✅ KNN entraîné

🎉 LES 3 MÉTHODES SONT PRÊTES ET SAUVEGARDÉES !


## Cellule 3 — Tester avec les préférences utilisateur


In [11]:
# PRÉFÉRENCES
budget_total = 1200
jours = 7
mois = 7
climat = "doux"  # chaud, doux, froid, indifferent
interets = ["culture", "gastronomy"]




import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

def recommander(budget_total, jours, mois, climat, interets, methode):

    if jours <= 0 or budget_total <= 0:
        raise ValueError("Le budget et la durée doivent être positifs.")

    data = df.copy()
    budget_jour = budget_total / jours

    # 1. Filtrer selon les contraintes utilisateur
    data = data[
        (data["month"] == int(mois)) &
        (data["budget_usd_day"] <= budget_jour)
    ].copy()

    temperatures = {
        "chaud": 29,
        "doux": 21,
        "froid": 9
    }

    temp_cible = temperatures[climat]

    if climat == "chaud":
        data = data[data["avg_temp_c"] >= 24]
    elif climat == "doux":
        data = data[data["avg_temp_c"].between(15, 27)]
    elif climat == "froid":
        data = data[data["avg_temp_c"] <= 16]

    if data.empty:
        return data

    # 2. Normaliser les activités sur une échelle 0-1
    activites = [
        "culture_score",
        "nature_score",
        "adventure_score",
        "gastronomy_score",
        "nightlife_score"
    ]

    valeurs = df[activites].astype(float)
    minimums = valeurs.min()
    amplitudes = (valeurs.max() - minimums).replace(0, 1)

    A = ((data[activites] - minimums) / amplitudes).clip(0, 1)

    # 3. Construire le profil utilisateur
    colonnes_choisies = [
        a + "_score"
        for a in interets
        if a + "_score" in activites
    ]

    if not colonnes_choisies:
        colonnes_choisies = activites

    # Similarité uniquement sur les activités choisies
    V = A[colonnes_choisies].to_numpy(dtype=float)
    profil = np.ones(len(colonnes_choisies))

    # 4. Scores des activités
    moyenne_activites = V.mean(axis=1)
    minimum_activites = V.min(axis=1)

    # Favoriser les destinations bonnes dans toutes
    # les activités sélectionnées
    score_activites = (
        0.7 * moyenne_activites +
        0.3 * minimum_activites
    )

    # 5. Scores climat et budget pour Weighted Score
    score_climat = np.clip(
        1 - np.abs(data["avg_temp_c"].to_numpy() - temp_cible) / 20,
        0, 1
    )

    score_budget = np.clip(
        1 - data["budget_usd_day"].to_numpy() / budget_jour,
        0, 1
    )

    # 6. Comparer les méthodes
    if methode == "Weighted Score":

        scores = (
            0.70 * score_activites +
            0.20 * score_climat +
            0.10 * score_budget
        )

    elif methode == "Cosine Similarity":

        scores = cosine_similarity(
            V, profil.reshape(1, -1)
        ).ravel()

        # Départager les destinations avec un score
        # cosinus similaire en favorisant les activités
        scores = scores * score_activites

    elif methode == "KNN":

        knn_local = NearestNeighbors(
            metric="manhattan",
            algorithm="brute"
        )

        knn_local.fit(V)

        distances, _ = knn_local.kneighbors(
            profil.reshape(1, -1),
            n_neighbors=len(V)
        )

        # Calculer directement la distance de chaque
        # destination au profil utilisateur
        distances = np.abs(V - profil).sum(axis=1)
        scores = 1 / (1 + distances)

    else:
        raise ValueError("Méthode inconnue")

    data["score"] = scores

    return (
        data.sort_values("score", ascending=False)
        .drop_duplicates(["country", "destination"])
        [
            [
                "country",
                "destination",
                "month",
                "score"
            ]
        ]
        .head(5)
        .reset_index(drop=True)
    )


## Cellule 4 — Ajouter les photos Unsplash



In [12]:
#tester 

import requests
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
UNSPLASH_KEY = user_secrets.get_secret("UNSPLASH_ACCESS_KEY")

def obtenir_photo_ville(ville, pays):
    url = "https://api.unsplash.com/search/photos"

    params = {
        "query": f"{ville} {pays} city travel",
        "per_page": 1,
        "orientation": "landscape"
    }

    headers = {
        "Authorization": f"Client-ID {UNSPLASH_KEY}"
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=15
    )
    response.raise_for_status()

    photos = response.json().get("results", [])

    if not photos:
        return None

    photo = photos[0]

    return {
        "image_url": photo["urls"]["regular"],
        "photographer": photo["user"]["name"],
        "photographer_url": photo["user"]["links"]["html"],
        "unsplash_url": photo["links"]["html"]
    }

photo = obtenir_photo_ville("Rome", "Italy")
print(photo)


{'image_url': 'https://images.unsplash.com/photo-1552832230-c0197dd311b5?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=M3wxMTMzNjM1fDB8MXxzZWFyY2h8MXx8Um9tZSUyMEl0YWx5JTIwY2l0eSUyMHRyYXZlbHxlbnwwfDB8fHwxNzk0Njc4OTc2fDA&ixlib=rb-4.1.0&q=80&w=1080', 'photographer': 'David Köhler', 'photographer_url': 'https://unsplash.com/@davidkhlr', 'unsplash_url': 'https://unsplash.com/photos/colosseum-arena-photography-VFRTXGw1VjU'}


Cette fonction fonctionne avec n'importe quelle destination renvoyée par ton modèle, tant que le résultat contient les colonnes destination et country.

In [13]:

def ajouter_photos_recommandations(top5):
    top5 = top5.copy()

    photos = []

    for _, row in top5.iterrows():
        ville = row["destination"]
        pays = row["country"]

        try:
            photo = obtenir_photo_ville(ville, pays)
        except Exception as e:
            print(f"Erreur photo pour {ville}: {e}")
            photo = None

        photos.append(photo or {})

    top5["image_url"] = [
        p.get("image_url") for p in photos
    ]

    top5["unsplash_url"] = [
        p.get("unsplash_url") for p in photos
    ]

    top5["photographer"] = [
        p.get("photographer") for p in photos
    ]

    top5["photographer_url"] = [
        p.get("photographer_url") for p in photos
    ]

    return top5


## cellule 4.1 — Tester et enregistrer 
tu ajoutes les photos aux résultats réels de ton modèle et tu exportes destinations_photos.json

In [14]:

import json
from pathlib import Path
import pandas as pd

def sauvegarder_profil(
    budget, jours, mois, climat, activites,
    destination, destinations_avec_photos=None
):
    # 1. Préférences de l'utilisateur
    profil = {
        "budget_total": float(budget),
        "jours": int(jours),
        "mois": int(mois),
        "climat": climat,
        "interets": activites,
        "destination_choisie": destination
    }

    dossier = Path("/kaggle/working")
    dossier.mkdir(parents=True, exist_ok=True)

    # 2. Enregistrer le profil utilisateur
    with open(dossier / "profil_utilisateur.json", "w", encoding="utf-8") as f:
        json.dump(profil, f, indent=4, ensure_ascii=False)

    # 3. Enregistrer les destinations et leurs photos
    if destinations_avec_photos is not None:
        if isinstance(destinations_avec_photos, pd.DataFrame):
            donnees = destinations_avec_photos.to_dict(orient="records")
        else:
            donnees = destinations_avec_photos

        with open(dossier / "destinations_photos.json", "w", encoding="utf-8") as f:
            json.dump(
                donnees,
                f,
                indent=4,
                ensure_ascii=False,
                default=str
            )

    return "Profil et résultats disponibles enregistrés avec succès !"


## Cellule 5 — Lancer Streamlit sur Kaggle

In [15]:
!pip install -q gradio


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [16]:

import gradio as gr
import pandas as pd
import json
from pathlib import Path
from html import escape

DOSSIER = Path("/kaggle/working")
DOSSIER.mkdir(parents=True, exist_ok=True)


# ====================================================
# 1. Générer les cartes HTML des destinations
# ====================================================

def creer_carte_photo(row, grande=False):

    ville = str(row["destination"])
    pays = str(row["country"])

    image_url = row.get("image_url")
    unsplash_url = row.get("unsplash_url")
    photographe = row.get("photographer")
    photographe_url = row.get("photographer_url")

    if pd.isna(image_url) or not image_url:
        return f"<p>Photo indisponible : {escape(ville)}</p>"

    image_url = escape(str(image_url), quote=True)

    # Attribution Unsplash
    credit = ""

    if pd.notna(photographe) and photographe:
        nom = escape(str(photographe))

        if pd.notna(photographe_url) and photographe_url:
            lien = escape(str(photographe_url), quote=True)
            credit = (
                f'<a href="{lien}" target="_blank">'
                f'{nom}</a> sur Unsplash'
            )
        else:
            credit = f"{nom} sur Unsplash"

    lien_photo = (
        escape(str(unsplash_url), quote=True)
        if pd.notna(unsplash_url) and unsplash_url
        else ""
    )

    hauteur = "320px" if grande else "180px"

    image = f"""
        <img src="{image_url}"
             style="width:100%;height:{hauteur};
                    object-fit:cover;border-radius:12px;">
    """

    if lien_photo:
        image = (
            f'<a href="{lien_photo}" target="_blank">'
            f'{image}</a>'
        )

    return f"""
    <div style="width:100%;padding:8px;">
        {image}
        <h3>{escape(ville)}, {escape(pays)}</h3>
        <small>{credit}</small>
    </div>
    """


def creer_galerie(df):

    if df.empty:
        return "<p>Aucune photo disponible.</p>"

    cartes = "".join(
        creer_carte_photo(row)
        for _, row in df.iterrows()
    )

    return f"""
    <div style="
        display:grid;
        grid-template-columns:repeat(
            auto-fit,minmax(220px,1fr)
        );
        gap:16px;
    ">
        {cartes}
    </div>
    """


# ====================================================
# 2. Recommandation + récupération des photos
# ====================================================

def demo(budget, jours, mois, climat, activites):

    resultats = []

    # Conserver les trois méthodes existantes
    for methode in [
        "Weighted Score",
        "Cosine Similarity",
        "KNN"
    ]:

        top5 = recommander(
            budget_total=budget,
            jours=int(jours),
            mois=int(mois),
            climat=climat,
            interets=activites,
            methode=methode
        )

        if not top5.empty:

            tableau = top5.copy()
            tableau.insert(0, "Méthode", methode)

            resultats.append(tableau)

    if not resultats:

        return (
            pd.DataFrame({
                "Message": ["Aucune destination trouvée"]
            }),
            gr.update(choices=[], value=None),
            [],
            [],
            "<p>Aucune photo disponible.</p>",
            "",
            "Aucune destination trouvée."
        )

    # Résultats des trois algorithmes
    tableau_final = pd.concat(
        resultats,
        ignore_index=True
    )

    # Destinations uniques, sans doublons
    destinations_uniques = (
        tableau_final[
            ["destination", "country"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Récupération automatique des photos
    destinations_photos = ajouter_photos_recommandations(
        destinations_uniques
    )

    # Liste des villes pour le dropdown
    # On utilise ville + pays pour éviter les ambiguïtés
    choix = [
        f"{row['destination']}, {row['country']}"
        for _, row in destinations_photos.iterrows()
    ]

    # Galerie des destinations
    galerie_html = creer_galerie(destinations_photos)

    # Conversion pour gr.State
    photos_data = (
        destinations_photos
        .where(pd.notna(destinations_photos), None)
        .to_dict(orient="records")
    )

    return (
        tableau_final,
        gr.update(choices=choix, value=None),
        choix,
        photos_data,
        galerie_html,
        "",
        "Recommandations générées avec succès !"
    )


# ====================================================
# 3. Afficher la destination sélectionnée
# ====================================================

def afficher_photo_selection(destination, photos_data):

    if not destination:
        return "<p>Sélectionnez une destination.</p>"

    for row in photos_data:

        identifiant = (
            f"{row['destination']}, {row['country']}"
        )

        if identifiant == destination:
            return creer_carte_photo(
                pd.Series(row),
                grande=True
            )

    return "<p>Photo indisponible.</p>"


# ====================================================
# 4. Sauvegarder profil + photos en JSON
# ====================================================

def sauvegarder_profil(
    budget, jours, mois, climat,
    activites, destination,
    destinations_valides, photos_data
):

    if not destination or destination not in destinations_valides:
        return "Veuillez choisir une destination recommandée."

    # Retrouver la ville réellement choisie
    destination_info = next(
        (
            row for row in photos_data
            if f"{row['destination']}, {row['country']}"
            == destination
        ),
        None
    )

    if destination_info is None:
        return "Destination introuvable."

    # Profil utilisateur
    profil = {
        "budget_total": float(budget),
        "jours": int(jours),
        "mois": int(mois),
        "climat": climat,
        "interets": activites,
        "destination_choisie": destination_info["destination"],
        "pays_destination": destination_info["country"]
    }

    # Enregistrer le profil
    with open(
        DOSSIER / "profil_utilisateur.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            profil,
            f,
            indent=4,
            ensure_ascii=False
        )

    # Enregistrer les destinations avec leurs photos
    with open(
        DOSSIER / "destinations_photos.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            photos_data,
            f,
            indent=4,
            ensure_ascii=False,
            default=str
        )

    return (
        f"Profil enregistré ! "
        f"Destination : {destination}"
    )


# ====================================================
# 5. Interface Gradio SmartTrip AI
# ====================================================

with gr.Blocks(title="SmartTrip AI") as app:

    gr.Markdown("# 🌍 SmartTrip AI")
    gr.Markdown(
        "### Recommandation intelligente de destinations"
    )

    with gr.Row():

        budget = gr.Number(
            label="Budget total ($)",
            value=1200
        )

        jours = gr.Number(
            label="Durée du voyage (jours)",
            value=7
        )

    with gr.Row():

        mois = gr.Dropdown(
            choices=list(range(1, 13)),
            value=7,
            label="Mois du voyage"
        )

        climat = gr.Dropdown(
            choices=["chaud", "doux", "froid"],
            value="doux",
            label="Climat préféré"
        )

    activites = gr.CheckboxGroup(
        choices=[
            "culture",
            "nature",
            "adventure",
            "gastronomy",
            "nightlife"
        ],
        value=["culture", "gastronomy"],
        label="Activités préférées"
    )

    bouton = gr.Button(
        "🔍 Recommander des destinations",
        variant="primary"
    )

    # Tableau original
    gr.Markdown("## Résultats des algorithmes")

    resultat = gr.Dataframe(
        label="Top 5 par méthode",
        interactive=False
    )

    # Nouvelle galerie
    gr.Markdown("## 📸 Photos des destinations recommandées")

    galerie = gr.HTML()

    # Choix utilisateur
    gr.Markdown("## Choisissez votre destination")

    destination_choisie = gr.Dropdown(
        choices=[],
        label="Destination recommandée"
    )

    gr.Markdown("## 🌍 Destination sélectionnée")

    photo_selectionnee = gr.HTML()

    # États internes
    destinations_valides = gr.State([])
    photos_data = gr.State([])

    # Bouton sauvegarde
    bouton_sauvegarder = gr.Button(
        "💾 Enregistrer mon choix",
        variant="primary"
    )

    message = gr.Textbox(
        label="Statut",
        interactive=False
    )

    # ====================================================
    # Événements
    # ====================================================

    bouton.click(
        fn=demo,
        inputs=[
            budget,
            jours,
            mois,
            climat,
            activites
        ],
        outputs=[
            resultat,
            destination_choisie,
            destinations_valides,
            photos_data,
            galerie,
            photo_selectionnee,
            message
        ]
    )

    # Afficher la photo après sélection
    destination_choisie.change(
        fn=afficher_photo_selection,
        inputs=[
            destination_choisie,
            photos_data
        ],
        outputs=photo_selectionnee
    )

    # Invalider les recommandations si les préférences changent
    def reinitialiser():
        return (
            pd.DataFrame(),
            gr.update(choices=[], value=None),
            [],
            [],
            "",
            "",
            ""
        )

    for champ in [
        budget,
        jours,
        mois,
        climat,
        activites
    ]:

        champ.change(
            fn=reinitialiser,
            outputs=[
                resultat,
                destination_choisie,
                destinations_valides,
                photos_data,
                galerie,
                photo_selectionnee,
                message
            ]
        )

    # Sauvegarde JSON
    bouton_sauvegarder.click(
        fn=sauvegarder_profil,
        inputs=[
            budget,
            jours,
            mois,
            climat,
            activites,
            destination_choisie,
            destinations_valides,
            photos_data
        ],
        outputs=message
    )


app.launch(share=True, inline=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1cd88eced69e953dc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
with open("/kaggle/working/profil_utilisateur.json", "r", encoding="utf-8") as f:
    profil = json.load(f)

print(profil)

{'budget_total': 1500.0, 'jours': 5, 'mois': 7, 'climat': 'doux', 'interets': ['nature', 'adventure'], 'destination_choisie': 'Geneva', 'pays_destination': 'Switzerland'}
